[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/19_gelu_solution.ipynb)

# 🟢 Solution: GELU Activation（参考版）

## 📋 题目描述

**难度：** Easy

实现 GELU 激活函数。

GELU（高斯误差线性单元）根据输入值平滑地进行门控，广泛用于 BERT 和 GPT 等 Transformer。

**签名:** `my_gelu(x) -> Tensor`

**参数:**
- `x` — 任意形状的输入张量

**返回:** 逐元素 GELU 激活，形状与输入相同

**约束:**
- 精确公式：`x * 0.5 * (1 + erf(x / sqrt(2)))`
- 必须与 `F.gelu` 误差在 1e-4 以内
- `gelu(0) = 0`

**提示：** 精确版：`0.5 * x * (1 + torch.erf(x / sqrt(2)))`
近似版：`0.5*x*(1+tanh(sqrt(2/π)*(x+0.044715*x³)))`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

import math

def my_gelu(x):
    # ---- Step 1: 计算 GELU 精确公式 ----
    # GELU(x) = x * Φ(x)，其中 Φ(x) 是标准正态分布的 CDF
    # Φ(x) = 0.5 * (1 + erf(x / sqrt(2)))
    # erf（误差函数）= (2/sqrt(π)) * ∫₀ˣ exp(-t²) dt
    #
    # 直觉理解：GELU 用"输入值有多大可能是正的"来门控
    #   x >> 0 时，erf → 1，GELU → x（完全通过）
    #   x << 0 时，erf → -1，GELU → 0（完全阻断）
    #   x ≈ 0 时，平滑过渡（不像 ReLU 在 0 点有拐角）
    #
    # 对比 ReLU：
    #   ReLU = max(0, x) — 硬门控，0 点不可导
    #   GELU = x * Φ(x) — 软门控，处处可导
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))


In [ ]:
x = torch.tensor([-2., -1., 0., 1., 2.])
print('GELU:', my_gelu(x))
print('F.gelu:', F.gelu(x))


In [ ]:
from torch_judge import check
check('gelu')


## 📝 核心思路总结

1. **GELU = x × Φ(x)**：用标准正态 CDF 作为门控信号
2. **erf 函数**：`Φ(x) = 0.5*(1 + erf(x/√2))`，PyTorch 内置 `torch.erf`
3. **软门控**：相比 ReLU 的硬门控，GELU 在零点附近平滑过渡
4. **近似版**：tanh 近似精度足够，计算更快
